In [ ]:
import asyncio
import time
from typing import Type
import os
import gc

from connector.binance.binance_perp import BinancePerp
from connector.binance.binance_spot import BinanceSpot
from connector.connector import BaseConnector
from connector.http_client import make_session
from connector.async_logger import AsyncLogger
from connector.utils import traceback_error_str
from connector.test_visualization import print_trades


class Monitor:
    def __init__(self, connector_type_list: list[Type[BaseConnector]]) -> None:
        self.logger = AsyncLogger("m").get_logger()
        self.connectors: dict[Type[BaseConnector], BaseConnector] = {}
        self.connector_type_list = connector_type_list

    async def loop_lag_monitor(self, period: float = 0.05, threshold_ms: float = 10):
        while True:
            next = time.perf_counter() + period
            next_t = time.thread_time() + period
            await asyncio.sleep(period)
            now = time.perf_counter()
            now_t = time.thread_time()

            lag_ms = max(0.0, (now - next) * 1000)
            lagt_ms = max(0.0, (now_t - next_t) * 1000)
            if lag_ms >= threshold_ms:
                self.logger.warning(f"event loop spike: {lag_ms:.1f} ms, thread: {lagt_ms:.1f} ms")

    async def run(self) -> None:
        try:
            async with make_session() as session:
                for connector_type in self.connector_type_list:
                    connector = connector_type(session=session, logger=self.logger)
                    self.connectors[connector_type] = connector
                async with asyncio.TaskGroup() as tg:
                    tg.create_task(self.loop_lag_monitor())

                    for connector in self.connectors.values():
                        tg.create_task(connector.run())
        except asyncio.CancelledError:
            self.logger.info("Monitor stopped (CancelledError)")
        except Exception:
            self.logger.error(traceback_error_str())


print(f"PID: {os.getpid()}")
connector_type_list: list[Type[BaseConnector]] = [BinancePerp, BinanceSpot]
m = Monitor(connector_type_list=connector_type_list)
gc.disable()
task = asyncio.create_task(m.run())


PID: 2442914


In [3]:
perp_symbols = set(instrument.unified_symbol for instrument in m.connectors[BinancePerp].instruments.values())
spot_symbols = set(instrument.unified_symbol for instrument in m.connectors[BinanceSpot].instruments.values())
unified_symbols = perp_symbols & spot_symbols
len(perp_symbols), len(spot_symbols), len(unified_symbols)

(542, 439, 371)

In [2]:
m.connectors[BinancePerp].orderbooks["BTCUSDT"].message

''

In [2]:
m.connectors[BinanceSpot].orderbooks["BTCUSDT"].message

'{"u":91193085088,"s":"BTCUSDT","b":"67026.03000000","B":"2.42203000","a":"67026.04000000","A":"0.24394000"}'

In [ ]:
m.connectors[BinanceSpot].trade_container["BTCUSDT"].public_trades

68300.37

In [4]:
m.connectors[BinanceSpot].trade_container["BTCUSDT"].public_trades_1s

In [4]:
min_delay = 0

In [ ]:
exchange_symbol = "BTCUSDT"
perp_ask_price: float = m.connectors[BinancePerp].orderbooks[exchange_symbol].best_ask
perp_bid_price: float = m.connectors[BinancePerp].orderbooks[exchange_symbol].best_bid
spot_bid_price: float = m.connectors[BinanceSpot].orderbooks[exchange_symbol].best_bid
spot_ask_price: float = m.connectors[BinanceSpot].orderbooks[exchange_symbol].best_ask
delay = int(time.time()*1000 - m.connectors[BinanceSpot].orderbooks[exchange_symbol].transaction_ts_ms)
delay2 = int(time.time()*1000 - m.connectors[BinancePerp].trade_container[exchange_symbol][-1].transaction_ts_ms)
if min_delay>delay:
    min_delay=delay
perp_ask_price, perp_bid_price, spot_bid_price, spot_ask_price, delay-min_delay, delay2-min_delay, min_delay

(0, 0, 0, 0, 76, 65, -1063)

In [13]:
m.connectors[BinancePerp].orderbooks.text_data.qsize()

0